# Steering 2.0 — direct residual-manifold steering

**Depends on `training_slowmo.ipynb`.** This notebook loads the final-step snapshot from `snapshots.pt`, the file produced by the slow-mo training run. Run that notebook first; by default the file lands at `./outputs/slowmo/snapshots.pt`.

**Standalone.** No Google Drive required. Runs anywhere PyTorch runs; a GPU keeps the 200 samples × 10 Δ × 6 methods loop fast (~minutes on a single T4 / L4).

---

**What we already know.** The algorithmic manifold lives at `pre_head` residual stream, grouped by `(la − lb) mod 96`. It is a clean Fourier circle (CV 0.07) — cleaner than W_U itself (0.09). The head reads from this manifold to produce per-token logits.

**The question this asks.** Previous steering rotated `h` on W_U Fourier planes (output-anchored, one circle per output token `c`). That's an indirect target — it modifies the head's *output-side* basis without directly moving `h` to the right point on the residual manifold.

**Steering 2.0 — use the right plane.** Steer `h` directly on the *residual manifold's* Fourier planes (input-side, anchored to `(la − lb)`). Six methods compared:

1. **δ-v2 baseline** — rotate `h` on W_U planes for 7 universal freqs (previous best).
2. **2.0a Direct manifold replacement:** `h_new = residual_manifold[(la − lb + Δ) mod 96]`. Replaces `h` entirely with the manifold's value at the target point.
3. **2.0b Residual Fourier rotation (universal k):** same algorithm as δ-v2 but on residual-manifold planes.
4. **2.0c Residual Fourier rotation (all 48 k):** same as 2.0b but use *every* frequency.
5. **2.0d Manifold replacement + deviation preservation:** `h_new = residual_manifold[target_t] + (h_actual − residual_manifold[current_t])`.
6. **2.0e Reversed-chirality rotation:** all 48 freqs but with `−Δθ` instead of `+Δθ`. Control: if the model's chirality matches our gauge, this method should *miss*.

**Outputs (`./outputs/steering2/`):**
- `hit_rates.png` — hit-target accuracy per Δ for all methods + achieved-vs-requested panel
- `summary.json` — aggregated hit rates and median achieved shifts per method per Δ

In [ ]:
# Cell 1 — Setup
# If running in Colab and you want I/O on Drive, uncomment:
#   from google.colab import drive
#   drive.mount('/content/drive')
# and point SLOMO_DIR / OUT_DIR at your Drive paths.

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, json, time, os, gc
from pathlib import Path
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.use_deterministic_algorithms(True, warn_only=True)

# Reads snapshots.pt produced by training_slowmo.ipynb (final entry).
# Run that notebook first; its default output is ./outputs/slowmo/snapshots.pt
SLOMO_DIR = Path('./outputs/slowmo')
OUT_DIR = Path('./outputs/steering2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

P = 97; PM1 = 96
D_MODEL = 384; N_HEADS = 12
BATCH = 1024
N_SAMPLES = 200
DELTAS = [-12, -8, -4, -2, -1, 1, 2, 4, 8, 12]

import matplotlib.pyplot as plt
BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.6,
    'axes.labelcolor': FG, 'axes.titlecolor': FG,
    'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.8,
})

def is_gen(g, p):
    seen = set(); x = 1
    for _ in range(p-1):
        x = (x*g) % p
        if x in seen: return False
        seen.add(x)
    return len(seen) == p-1
for g in range(2, P):
    if is_gen(g, P): break
exp_table = np.zeros(PM1, dtype=np.int64); x = 1
for t in range(PM1): exp_table[t] = x; x = (x*g) % P
dlog = np.zeros(P, dtype=np.int64)
for t in range(PM1): dlog[exp_table[t]] = t
print(f'g={g}, OUT={OUT_DIR.resolve()}')


In [ ]:
# Cell 2 — Model + data + load final endpoint
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.dropout(self.w2(gate * self.w3(h2)))

class GrokModel(nn.Module):
    def __init__(self, p=97, d=384, nh=12):
        super().__init__()
        self.tok_emb = nn.Embedding(p+2, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(2)])
        self.norm_f = RMSNorm(d); self.head = nn.Linear(d, p, bias=False); self.p = p
    def forward_capture_pre_head(self, a, b):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        h = self.norm_f(x)[:, -1, :]
        return self.head(h), h
    def head_only(self, h):
        return self.head(h)

all_a, all_b, all_c = [], [], []
for a in range(P):
    for b in range(1, P):
        all_a.append(a); all_b.append(b); all_c.append((a * pow(b, P-2, P)) % P)
all_a = torch.tensor(all_a, device=device)
all_b = torch.tensor(all_b, device=device)
all_c = torch.tensor(all_c, device=device)

# diff_arr: la-lb mod 96 for each input (a, b), -1 if a=0
diff_arr = np.array([
    -1 if int(all_a[i].item()) == 0 else (int(dlog[int(all_a[i].item())]) - int(dlog[int(all_b[i].item())])) % PM1
    for i in range(len(all_a))
])

# Load final snapshot from Plan 1b
print('Loading final snapshot...')
snapshots = torch.load(SLOMO_DIR / 'snapshots.pt', map_location='cpu', weights_only=False)
final_sd = {k: v.float() for k, v in snapshots[-1]['state_dict'].items()}
del snapshots; gc.collect()

m = GrokModel(P, D_MODEL, N_HEADS).to(device)
m.load_state_dict({k: v.to(device) for k, v in final_sd.items()}, strict=False); m.eval()

# Sanity
with torch.no_grad():
    logits, _ = m.forward_capture_pre_head(all_a, all_b)
    acc = (logits.argmax(-1) == all_c).float().mean().item()
print(f'Model loaded, full-dataset acc = {acc:.4f}')


In [ ]:
# Cell 3 — Compute residual manifold (the actual algorithmic substrate)
@torch.no_grad()
def compute_residual_manifold(model):
    out = np.zeros((PM1, D_MODEL)); counts = np.zeros(PM1)
    for i in range(0, len(all_a), BATCH):
        ab = all_a[i:i+BATCH]; bb = all_b[i:i+BATCH]
        _, h = model.forward_capture_pre_head(ab, bb)
        h_np = h.cpu().numpy()
        for j in range(len(ab)):
            di = diff_arr[i + j]
            if di < 0: continue
            out[di] += h_np[j]; counts[di] += 1
    out /= np.maximum(counts[:, None], 1)
    return out  # [96, d_model]

residual_manifold = compute_residual_manifold(m)
print(f'Residual manifold shape: {residual_manifold.shape}')

# Center the manifold (subtract mean — removes constant offset)
residual_mean = residual_manifold.mean(axis=0)  # [d_model]
manifold_centered = residual_manifold - residual_mean  # [96, d_model]

# Fourier decompose along the 96-axis
def fourier_decompose_dlog(M):
    p = M.shape[0]; K = p//2 + 1
    i = np.arange(p)
    A = np.zeros((K, M.shape[1])); B = np.zeros((K, M.shape[1]))
    for k in range(K):
        c = np.cos(2*np.pi*k*i/p); s = np.sin(2*np.pi*k*i/p)
        norm = p/2 if k != 0 and k != p//2 else p
        A[k] = c @ M / norm; B[k] = s @ M / norm
    power = (A**2).sum(1) + (B**2).sum(1)
    return A, B, power

A_resid, B_resid, power_resid = fourier_decompose_dlog(manifold_centered)
p_no_dc = power_resid.copy(); p_no_dc[0] = 0
top_universal = np.argsort(p_no_dc)[::-1][:7].tolist()
all_freqs = list(range(1, PM1//2 + 1))
print(f'Top-7 universal freqs in residual manifold: {top_universal}')
print(f'Total non-DC freqs: {len(all_freqs)}')

# Build orthonormal 2D bases for each freq
def make_basis(a, b):
    u = a / (np.linalg.norm(a) + 1e-12)
    v = b - (b @ u) * u
    v_norm = np.linalg.norm(v)
    if v_norm < 1e-12:
        # degenerate: pick something orthogonal
        v = np.eye(len(u))[0] - u[0]*u; v_norm = np.linalg.norm(v)
    v = v / (v_norm + 1e-12)
    return u, v, np.linalg.norm(a), v_norm  # return original norms too

bases = {}  # k -> (u, v, radius)
for k in all_freqs:
    u, v, _, _ = make_basis(A_resid[k], B_resid[k])
    bases[k] = (u, v)
print('Bases built for all 48 frequencies')


In [ ]:
# Cell 4 — Five steering methods

def rotate_on_plane(h, u, v, dtheta):
    """Rotate h's projection on (u, v) by dtheta. Preserves all other dims."""
    x = h @ u
    y = h @ v
    new_x = x * np.cos(dtheta) - y * np.sin(dtheta)
    new_y = x * np.sin(dtheta) + y * np.cos(dtheta)
    return h + (new_x - x) * u + (new_y - y) * v

def steer_dv2_baseline(h, current_t, target_t, wu_bases):
    """δ-v2: rotate h on W_U Fourier planes for 7 universal freqs."""
    delta = (target_t - current_t) % PM1
    if delta > PM1 // 2: delta -= PM1
    h_new = h.copy()
    for k, (u, v) in wu_bases.items():
        dtheta = 2 * np.pi * k * delta / PM1
        h_new = rotate_on_plane(h_new, u, v, dtheta)
    return h_new

def steer_2a_direct(h, current_t, target_t):
    """Replace h entirely with manifold[target_t]."""
    return residual_manifold[target_t].copy()

def steer_2b_residual_universal(h, current_t, target_t):
    """Fourier rotation on residual planes, 7 universal frequencies."""
    delta = (target_t - current_t) % PM1
    if delta > PM1 // 2: delta -= PM1
    h_new = h.copy()
    for k in top_universal:
        u, v = bases[k]
        dtheta = 2 * np.pi * k * delta / PM1
        h_new = rotate_on_plane(h_new, u, v, dtheta)
    return h_new

def steer_2c_residual_all(h, current_t, target_t):
    """Fourier rotation on residual planes, ALL 48 frequencies."""
    delta = (target_t - current_t) % PM1
    if delta > PM1 // 2: delta -= PM1
    h_new = h.copy()
    for k in all_freqs:
        u, v = bases[k]
        dtheta = 2 * np.pi * k * delta / PM1
        h_new = rotate_on_plane(h_new, u, v, dtheta)
    return h_new

def steer_2d_manifold_preserve(h, current_t, target_t):
    """Replace manifold portion with target, keep deviation from current manifold point."""
    deviation = h - residual_manifold[current_t]
    return residual_manifold[target_t] + deviation

def steer_2e_reversed(h, current_t, target_t):
    """Chirality test: apply REVERSED rotation on residual planes, all 48 freqs.
    If the model's chirality is flipped relative to our gauge, this should hit target
    while normal-direction rotation misses. If hit rate is near zero, chirality is consistent."""
    delta = (target_t - current_t) % PM1
    if delta > PM1 // 2: delta -= PM1
    h_new = h.copy()
    for k in all_freqs:
        u, v = bases[k]
        dtheta = -2 * np.pi * k * delta / PM1  # NEGATIVE — reversed rotation
        h_new = rotate_on_plane(h_new, u, v, dtheta)
    return h_new

# Build W_U-plane bases for δ-v2 baseline (same as in original δ-v2)
head_weight = final_sd['head.weight'].cpu().numpy()  # [P, d]
head_dlog = head_weight[exp_table]  # [96, d]
head_centered = head_dlog - head_dlog.mean(axis=0)
A_wu, B_wu, power_wu = fourier_decompose_dlog(head_centered)
p_no_dc_wu = power_wu.copy(); p_no_dc_wu[0] = 0
wu_top7 = np.argsort(p_no_dc_wu)[::-1][:7].tolist()
wu_bases = {}
for k in wu_top7:
    u, v, _, _ = make_basis(A_wu[k], B_wu[k])
    wu_bases[k] = (u, v)
print(f'W_U top-7 freqs (for baseline): {wu_top7}')


In [ ]:
# Cell 5 — Run experiment: 200 samples × 10 Δ × 5 methods
rng = np.random.default_rng(42)
valid_idx = np.where(diff_arr >= 0)[0]
sample_idx = rng.choice(valid_idx, size=N_SAMPLES, replace=False)

sa = all_a[sample_idx]; sb = all_b[sample_idx]; sc = all_c[sample_idx]
sample_current_t = diff_arr[sample_idx]  # current (la - lb) for each sample

# Capture h_pre_head for all samples
with torch.no_grad():
    base_logits, h_batch = m.forward_capture_pre_head(sa, sb)
base_pred = base_logits.argmax(-1).cpu().numpy()
sc_np = sc.cpu().numpy()
correct_mask = (base_pred == sc_np)
n_correct = int(correct_mask.sum())
print(f'Base predictions correct: {n_correct} / {N_SAMPLES}')

h_np = h_batch.cpu().numpy()

methods = {
    'dv2_baseline_wu': lambda h, ct, tt: steer_dv2_baseline(h, ct, tt, wu_bases),
    '2a_direct':       lambda h, ct, tt: steer_2a_direct(h, ct, tt),
    '2b_resid_univ':   lambda h, ct, tt: steer_2b_residual_universal(h, ct, tt),
    '2c_resid_all':    lambda h, ct, tt: steer_2c_residual_all(h, ct, tt),
    '2d_manifold_pres': lambda h, ct, tt: steer_2d_manifold_preserve(h, ct, tt),
    '2e_reversed':     lambda h, ct, tt: steer_2e_reversed(h, ct, tt),
}

results = defaultdict(lambda: defaultdict(list))
t0 = time.time()
for delta in DELTAS:
    for i in range(N_SAMPLES):
        if not correct_mask[i]: continue
        current_t = int(sample_current_t[i])
        target_t = (current_t + delta) % PM1
        target_c = int(exp_table[target_t])
        h_orig = h_np[i].astype(np.float32)
        for method_name, fn in methods.items():
            h_new = fn(h_orig, current_t, target_t)
            with torch.no_grad():
                logits = m.head_only(torch.tensor(h_new, device=device, dtype=torch.float32).unsqueeze(0))
                pred = int(logits.argmax(-1).item())
            hit = (pred == target_c)
            shift = None
            if pred != 0:
                s = (dlog[pred] - current_t) % PM1
                if s > PM1 // 2: s -= PM1
                shift = s
            results[method_name][delta].append({'hit': hit, 'shift': shift, 'pred': pred, 'target': target_c})
    print(f'Δ={delta:>3}: ' + '  '.join(
        f'{m_name}={np.mean([r["hit"] for r in results[m_name][delta]]):.2f}'
        for m_name in methods))
print(f'\nTotal time: {time.time()-t0:.1f}s')


In [ ]:
# Cell 6 — Plot hit rates and achieved shifts
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = {
    'dv2_baseline_wu': '#6E6E6E',
    '2a_direct': '#D45A2A',
    '2b_resid_univ': '#3A6E8C',
    '2c_resid_all': '#3A8C3A',
    '2d_manifold_pres': '#7A4DA8',
    '2e_reversed':    '#A83A6E',
}
labels = {
    'dv2_baseline_wu': 'δ-v2 (W_U planes, 7 freqs) — baseline',
    '2a_direct': '2.0a: direct manifold replacement',
    '2b_resid_univ': '2.0b: residual Fourier, 7 universal',
    '2c_resid_all': '2.0c: residual Fourier, ALL 48 freqs',
    '2d_manifold_pres': '2.0d: manifold replacement + deviation',
    '2e_reversed':    '2.0e: REVERSED rotation (chirality test)',
}

ax = axes[0]
for m_name in methods:
    hits = [np.mean([r['hit'] for r in results[m_name][d]]) for d in DELTAS]
    ax.plot(DELTAS, hits, '-o', color=colors[m_name], lw=2, markersize=5,
            label=labels[m_name], alpha=0.9)
ax.set_xlabel('requested dlog shift Δ')
ax.set_ylabel('hit-target accuracy')
ax.set_title('Steering 2.0: hit rate per Δ')
ax.set_ylim(-0.02, 1.05)
ax.axhline(1/97, color=MUTED, ls=':', lw=0.5, alpha=0.5, label='random chance')
ax.legend(fontsize=8, frameon=False, loc='lower center')

ax = axes[1]
for m_name in methods:
    mean_shifts = []
    for d in DELTAS:
        shifts = [r['shift'] for r in results[m_name][d] if r['shift'] is not None]
        mean_shifts.append(np.mean(shifts) if shifts else 0)
    ax.plot(DELTAS, mean_shifts, '-o', color=colors[m_name], lw=2, markersize=5,
            label=labels[m_name].split(' — ')[0].split(': ')[0], alpha=0.85)
ax.plot(DELTAS, DELTAS, ':', color=MUTED, lw=1, alpha=0.5, label='ideal')
ax.set_xlabel('requested dlog shift Δ')
ax.set_ylabel('mean achieved shift')
ax.set_title('Achieved vs requested')
ax.legend(fontsize=8, frameon=False)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(OUT_DIR / 'hit_rates.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 7 — Summary
summary = {
    'config': {'P': P, 'PM1': PM1, 'g': int(g), 'n_samples': N_SAMPLES, 'deltas': DELTAS,
                'wu_top7': wu_top7, 'residual_top7': top_universal,
                'all_freqs_count': len(all_freqs)},
    'aggregate': {},
}
print(f'\n{"method":<25} {"mean hit rate":>15} {"median achieved":>18}')
print('-' * 65)
for m_name in methods:
    all_hits = []; all_shifts = []
    per_delta = {}
    for d in DELTAS:
        hits = [r['hit'] for r in results[m_name][d]]
        shifts = [r['shift'] for r in results[m_name][d] if r['shift'] is not None]
        all_hits.extend(hits)
        all_shifts.extend(shifts)
        per_delta[str(d)] = {
            'hit_rate': float(np.mean(hits)),
            'mean_shift': float(np.mean(shifts)) if shifts else None,
            'median_shift': float(np.median(shifts)) if shifts else None,
        }
    summary['aggregate'][m_name] = {
        'overall_hit_rate': float(np.mean(all_hits)),
        'per_delta': per_delta,
    }
    print(f'{m_name:<25} {np.mean(all_hits)*100:>13.1f}% {np.median(all_shifts):>17.1f}')

with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved {OUT_DIR / "summary.json"}')


## How to read Steering 2.0

**`hit_rates.png` left panel.** Hit-target accuracy per Δ for all 5 methods. The δ-v2 baseline (grey) should match the ~50% we got before; the new methods (coloured) should improve over it.

**Expected ordering, from worst to best:**
1. *δ-v2 baseline* (~50%): W_U-plane rotation, indirect target.
2. *2.0b* (~60-75%): same algorithm, but on residual manifold planes (the right coordinate system).
3. *2.0c* (~75-90%): residual Fourier with all 48 frequencies, maximum resolution.
4. *2.0a* (~85-99%): direct manifold replacement, assumes manifold IS sufficient.
5. *2.0d* (~90-99%): manifold replacement + per-input deviation preservation, best of both.

**Right panel.** Mean achieved shift vs requested. Ideal diagonal. The methods that hit target reliably should track the diagonal closely.

## Verdicts

- *If 2.0a hits 95%+:* the residual manifold IS the algorithm. The head reads it almost perfectly. Manifold replacement is sufficient steering.
- *If 2.0a hits 70-85%:* manifold is good but not the only signal — head also uses non-manifold components.
- *If 2.0d > 2.0a:* per-input deviation matters; the head conditions on noise beyond the manifold.
- *If 2.0c > 2.0a:* Fourier rotation preserves more useful structure than direct replacement.
- *If 2.0b ≈ δ-v2:* residual planes give same result as W_U planes — they're not in fact different coordinate systems for this task.

If we cross 80% in any method, we've shown the algorithm is *steerable to high precision*, validating both the manifold geometry and the mechanistic interpretation. That's paper-level.